# **MD Simulation Colab - classic MD with DNA without ZEN**

This branch purpose is to simulate the duplexes developped during the Sorbonne iGEM's 2026 team projet PUREGRAIN-ZEN. The goal here is to observe if a duplex remains joined, reorders itself, presents conformational changes or separates itself in a classic MD using water an ions. It was originally made as a notebook working in google colab and was adapted to suit local computational needs. The following notebook gathers all the necessary functions calls needed for analysis

This Notebook is an attempt to adapt and expand upon the work previously done by Alexis Fahy, based on a cloud-based MD workflow, ["Making it Rain"](https://github.com/pablo-arantes/making-it-rain). I also thank Halil Arda for collaborating with me in this endeavor and all other contributors who may appear in the future

Along with this notebook you should also have received a folder containing the architecture and files needed to run or test everything. Keep it as is - or change it accordingly if you're familiar with the concept enough.

We start by setting up a GPU environment to work with. On a free colab account you should be able to get a **T4**

In [ ]:
# Setting GPU is important later down the line - check its availability here
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Activate the GPU via Runtime > Change runtime type, and relaunch this cell.')
else:
  print(gpu_info)

Setup your drive to work. To keep it simple, You should have installed the folder sent with the notebook at the root of your drive (or any other path if you know how to rewrite path and if you'd rather do it your own way). The function `set_dir()` automatically creates a connection to your drive

If you're running this notebook locally, you may skip this section


In [3]:
import colab_funcs
import constants

In [ ]:
colab_funcs.set_dir()

# **Step 1: Building the system**

In [ ]:
import system_builder
import constants

In [9]:
system_builder.build_system(constants.input_pdb_ext, constants.Z1_ext_res_folder, fit_box=False)

  strand A: 5'-GACAGTTTAGCTGGTATAATTAGATCC-3' (27 nt, residues 1-27)
  strand B: 5'-CTACCAGCTTTGAGGCTCGATCCAGCTTATTCAATTATACCAGCTTATTCAATTATACCAGC-3' (62 nt, residues 28-89)
saved 115,547 atoms -> /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/system.xml, /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/model.cif
solute 8.18 nm across, shortest image distance 10.91 nm, worst-case gap 2.73 nm


(<openmm.openmm.System; proxy of <Swig Object of type 'OpenMM::System *' at 0x332880030> >,
 <openmm.app.modeller.Modeller at 0x32ccec6b0>)

# **Step 2: Minimize and Balance the system**
The purpose of this operation is to minimize and balance the system.

Like the previous Step, this section aims to be independent, considering the possibility that Colab may crash anytime or exceed GPU hours. To start it, you'll need the system and model files that are generated from step 1.

In order to proceed with equilibration, we defined a `Equilibrator` which takes in a ladder and the equilibration pipeline. The ladder serves as a guide to dictate what kind of transformations should be applied to the system during equilibration

In [6]:
from equilibration import Equilibrator
import constants

In [7]:
#The default ladder parameter can be seen by calling the ladder property of the class
Equilibrator().ladder

[('min', 1000, 0),
 ('min', 0, 0),
 ('heat', 1000, 0.2),
 ('npt', 500, 0.3),
 ('npt', 0, 5)]

In [8]:
#You can also pass a custom made ladder to suit your need from class instanciation alone
Equilibrator([("min", 0, 0)]).ladder

[('min', 0, 0)]

In [ ]:
#Call the equilibrate function of the Equilibrator to start production 
simulation = Equilibrator().equilibrate(constants.Z1_ori_res_folder)

# **Step 3: Running production**

Now lies the heart of what we're trying to accomplish, simulating the behavior of the lone duplex. We are, however limited by one thing: computationnal ressources. In order to deal with this in colab, The functions have been designed to organize a workflow between local colab saves and transfers towards a google drive. This, way, multiple accounts can assist with the processing step

One thing to note about processing and saving files: the frequency at which files have to be updated towards the google drive appears so high that it the i/o can be **bugged**: while attempting direct saves towards a google drive, I noticed that the .xtc file i was exporting (which basically contains what we wish to observe) was not being kept up to date with the colab simulation. The workaround found was to work in the local colab env, then set timers which will help flush data towards the Drive and prevent corrupted saves

## **Running for one production**

In [1]:
from production import produce
import constants

In [3]:
produce(constants.Z1_ext_res_folder, save_name="Z1_Z0_test_run_short_ns", ns=0.01, save_ps=1)

Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
  trajectory: 2,985 of 90,577 atoms (water excluded)
running 0.0 ns of 0.01 ns

  COMPLETE: 0.01 ns at 5 ns/day


## **Running for replicates**

In [1]:
from replicates import run_replicas
from equilibration import Equilibrator
import constants

SESSION ID:macbookpro.home-6a934b


In [2]:
run_replicas(constants.Z1_ext_res_folder, 
             local_root=constants.Z1_ext_res_folder + "/replicas",
             save_ps=1, ns=0.01,
             equil=Equilibrator([('min', 200, 0)]),
             cleanup_local=True)

solute 8.17 nm across, shortest image distance 6.93 nm, worst-case gap -1.25 nm   !! below the cutoff
r01: equilibrating (seed 1001)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
equilibrating: 1,806 restrained heavy atoms, CPU
ladder:
  min   k=   200      0 ns   sigma=1.14 A
      k=200 call   200  E=   -1514279.5  maxF=   1400.8
      k=200 call   400  E=   -1540568.4  maxF=   3141.1
      k=200 call   600  E=   -1550553.2  maxF=    997.2
      k=200 call   800  E=   -1555258.0  maxF=   1248.6
      k=200 call  1000  E=   -1288964.1  maxF=  12489.0
      k=200 call  1200  E=   -1377968.9  maxF=   6625.7
      k=200 call  1400  E=   -1410944.8  maxF=   6041.4
      k=200 call  1600  E=   -1426445.5  maxF=   5915.8
      k=200 call  1800  E=   -1436380.5  maxF=   3915.0
      k=200 call  2000  E=   -1442612.4  maxF=   3047.9
      k=200 call  2200  E=   -1447005.8  maxF=   2570.6
      k=200 call  2400  E=   -1451033.4  maxF=  10234.9
      k=200 call  2600  E=

/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/equilibration.py:200: UserWarning: genfromtxt: Empty input file: "/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/Z1_ext/r01/equil.csv"
  rows = np.atleast_1d(np.genfromtxt(csv, delimiter=",", names=True, deletechars=" "))


r01: production (seed 1501)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
  trajectory: 2,985 of 90,577 atoms (water excluded)
running 0.0 ns of 0.01 ns
  wall-clock limit 2.8851532814237806 h (stopping 15 min early to save cleanly)
  [sync] 22.6 MB in 0 s (651 MB/s) -> /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/r01

  COMPLETE: 0.01 ns at 5 ns/day
r01: complete
r02: equilibrating (seed 1002)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
equilibrating: 1,806 restrained heavy atoms, CPU
ladder:
  min   k=   200      0 ns   sigma=1.14 A
      k=200 call   200  E=   -1514453.8  maxF=   1638.1
      k=200 call   400  E=   -1540308.7  maxF=    832.8
      k=200 call   600  E=   -1550228.1  maxF=    862.0
      k=200 call   800  E=   -1555424.1  maxF=   1397.9
      k=200 call  1000  E=   -1311508.9  maxF=  21398.7
      k=200 call  1200  E=   -1386378.2  maxF=   6410.5
      k=200 call  1400  E=   -1412018.3 

/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/equilibration.py:200: UserWarning: genfromtxt: Empty input file: "/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/Z1_ext/r02/equil.csv"
  rows = np.atleast_1d(np.genfromtxt(csv, delimiter=",", names=True, deletechars=" "))


  (log too short to assess density and drift)
  [sync] 59.1 MB in 0 s (477 MB/s) -> /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/r02
r02: production (seed 1502)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
  trajectory: 2,985 of 90,577 atoms (water excluded)
running 0.0 ns of 0.01 ns
  wall-clock limit 2.7357219663593506 h (stopping 15 min early to save cleanly)
  [sync] 22.6 MB in 0 s (498 MB/s) -> /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/r02

  COMPLETE: 0.01 ns at 5 ns/day
r02: complete
r03: equilibrating (seed 1003)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
equilibrating: 1,806 restrained heavy atoms, CPU
ladder:
  min   k=   200      0 ns   sigma=1.14 A
      k=200 call   200  E=   -1513933.7  maxF=   2027.6
      k=200 call   400  E=   -1539880.3  maxF=   2131.3
      k=200 call   600  E=   -1550138.3  maxF=   1946.3
      k=200 call   800  E=   -1555090.5  m

/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/equilibration.py:200: UserWarning: genfromtxt: Empty input file: "/Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/Z1_ext/r03/equil.csv"
  rows = np.atleast_1d(np.genfromtxt(csv, delimiter=",", names=True, deletechars=" "))


r03: production (seed 1503)
Error: No compatible OpenCL platform is available
PROCEEDING WITH CPU
  trajectory: 2,985 of 90,577 atoms (water excluded)
running 0.0 ns of 0.01 ns
  wall-clock limit 2.5703106507990094 h (stopping 15 min early to save cleanly)
  [sync] 22.6 MB in 0 s (740 MB/s) -> /Users/karlkoffi/Documents/GitHub/MD_simulation_Colab/results/Z1_ext/replicas/r03

  COMPLETE: 0.01 ns at 5 ns/day
r03: complete


# **Step 4: Analysis**

To analyze your trajectories, you have three functions available to you:
- `analyze_run`: takes a production runs a gives a detailed series of metrics, see the table metric file. This function is also useful if you wish to study the state of a particular system post equilibration 

- `analyze_fraying`: taks a production run and attempts to study fraying effects. Metrics produced detailed in the table metrics file

- `analyze replicas`: leverages both `analyze_run` and `analyze_fraying` to studies batches of replicas together. Additionnal produced results detailed in the table metric file

In [3]:
from evaluation import analyze_run, analyze_fraying, analyze_replicas
import constants

In [4]:
analyze_replicas(constants.Z1_ext_res_folder, traj="md")


 r01 
  arm: 61 nt (A:DG1 ... B:DT75), neighbours of either partner within 2 nt ignored
r01/md: 0.009 ns, 2 strand(s), 31 native pairs (16 inter-strand)
means over the last 50%:
q_native           0.574
q_inter            0.538
q_intra            0.613
q_native_hb1       0.645
nonnative_pairs    0.600
rmsd_nm            0.167
rmsd_core_nm       0.147
rg_nm              2.501
a_form             0.591
cations_5A         6.000
e2e_A_nm           7.294
e2e_B_nm           4.268
com_AB_nm          1.904
min_image_nm         NaN
arm_contacts       7.000

native pairs that are weak or absent from the model:
         pair type  kind  native   end  in_model  occupancy  occ_hb1  d1_A  open_events  open_events_hb1  longest_open_ns  arm_contact  contact_frames  occ_arm_on  occ_arm_off
A:DG14-B:DC86   GC inter    True  True      True        0.0      0.1  3.77            0                0            0.010          0.0             0.0         NaN          NaN
A:DA18-B:DT81   AT inter    True False  

(                     occupancy_mean  occupancy_sem  p_open_mean  p_open_sem  \
 pair          end                                                             
 A:DG10-B:DC89 True           0.9333         0.0667       0.0000      0.0000   
 B:DG88-A:DC11 False          0.9667         0.0333       0.0000      0.0000   
 B:DA87-A:DT12 False          0.9000         0.0000       0.0000      0.0000   
 A:DG13-B:DC86 False          1.0000         0.0000       0.0000      0.0000   
 A:DG14-B:DC85 False          0.9667         0.0333       0.0000      0.0000   
 A:DG14-B:DC86 True           0.0000         0.0000       0.9333      0.0667   
 B:DA84-A:DT15 False          0.9667         0.0333       0.0000      0.0000   
 A:DA16-B:DT83 False          0.9000         0.0577       0.0000      0.0000   
 B:DA82-A:DT17 False          0.8000         0.1528       0.0000      0.0000   
 A:DA18-B:DT81 False          0.0000         0.0000       1.0000      0.0000   
 A:DA19-B:DT80 False          0.0000    